# In-Context Learning Transformer

This notebook trains a transformer to perform in-context linear regression.

## Setup

In [ ]:
# Check GPU availability
import torch

print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"CUDA Capability: {torch.cuda.get_device_capability(0)}")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > Select GPU")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set up paths - adjust PROJECT_PATH to where your files are in Google Drive
import sys
import os

# Path to the ICL folder in your Google Drive
PROJECT_PATH = '/content/drive/MyDrive/micro-research/ICL'

# Alternative: if you cloned to /content/
# PROJECT_PATH = '/content/micro-research/ICL'

# Add to Python path so we can import the modules
if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

os.chdir(PROJECT_PATH)
print(f"Working directory: {os.getcwd()}")
print(f"Files: {os.listdir('.')}")

In [ ]:
# Import our modules
from data import make_batch_xy, make_batch_xy_padded, make_batch_xy_fixed_padded
from model import ICLTransformer, create_model
from eval import ols_predict_from_seq, sweep_context_lengths, eval_suite

import torch
import torch.nn.functional as F

print("Imports successful!")

## Configuration

In [ ]:
# Hyperparameters
CONFIG = {
    # Data
    'd': 5,              # Input dimension
    'n_ctx_max': 128,    # Maximum context length
    'n_ctx_min': 2,      # Minimum context length
    'noise_std': 0.05,   # Label noise
    
    # Model
    'd_model': 256,      # Hidden dimension
    'n_heads': 8,        # Attention heads
    'n_layers': 6,       # Transformer layers
    
    # Training
    'batch_size': 256,
    'steps': 15000,
    'lr': 3e-4,
    'weight_decay': 1e-2,
    'log_every': 500,
}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## Create Model

In [ ]:
model = create_model(
    d=CONFIG['d'],
    n_ctx_max=CONFIG['n_ctx_max'],
    d_model=CONFIG['d_model'],
    n_heads=CONFIG['n_heads'],
    n_layers=CONFIG['n_layers'],
    device=device,
)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

## Training

In [ ]:
opt = torch.optim.AdamW(
    model.parameters(), 
    lr=CONFIG['lr'], 
    weight_decay=CONFIG['weight_decay']
)

# Use bfloat16 on Ampere+ GPUs (A100, etc.)
use_bf16 = (
    torch.cuda.is_available() 
    and torch.cuda.get_device_capability(0)[0] >= 8
)
dtype = torch.bfloat16 if use_bf16 else torch.float16
print(f"Using {'bfloat16' if use_bf16 else 'float16'} mixed precision")

In [ ]:
# Training loop
losses = []

for step in range(1, CONFIG['steps'] + 1):
    seq, yq, pad_mask, n_ctx = make_batch_xy_padded(
        batch_size=CONFIG['batch_size'],
        d=CONFIG['d'],
        n_ctx_max=CONFIG['n_ctx_max'],
        n_ctx_min=CONFIG['n_ctx_min'],
        noise_std=CONFIG['noise_std'],
    )
    seq = seq.to(device)
    yq = yq.to(device)
    pad_mask = pad_mask.to(device)

    opt.zero_grad(set_to_none=True)
    
    with torch.amp.autocast(device, dtype=dtype):
        pred = model(seq, pad_mask=pad_mask)
        loss = F.mse_loss(pred, yq)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    
    losses.append(loss.item())

    if step % CONFIG['log_every'] == 0:
        print(f"step {step:5d} | train RMSE: {loss.sqrt().item():.4f} | mean n_ctx: {n_ctx.float().mean().item():.1f}")

In [ ]:
# Plot training loss
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(losses, alpha=0.3)
# Smoothed
window = 100
smoothed = [sum(losses[max(0,i-window):i+1])/min(i+1, window) for i in range(len(losses))]
plt.plot(smoothed, label='smoothed')
plt.xlabel('Step')
plt.ylabel('MSE Loss')
plt.title('Training Loss')
plt.legend()
plt.show()

## Save Checkpoint

In [ ]:
# Save model to Google Drive
checkpoint_path = os.path.join(PROJECT_PATH, 'model_checkpoint.pt')
torch.save({
    'model_state_dict': model.state_dict(),
    'config': CONFIG,
}, checkpoint_path)
print(f"Saved checkpoint to {checkpoint_path}")

## Evaluation

In [ ]:
# Context length sweep: compare model vs OLS
print("Context Length Sweep")
print("=" * 50)

n_ctx_list = [2, 4, 8, 12, 16, 24, 32, 48, 64, 96, 128]
results = sweep_context_lengths(
    model,
    d=CONFIG['d'],
    n_ctx_list=n_ctx_list,
    batch_size=4096,
    n_ctx_max=CONFIG['n_ctx_max'],
    device=device,
)

In [ ]:
# Plot model vs OLS performance
n_vals = [r[0] for r in results]
model_rmse = [r[1] for r in results]
ols_rmse = [r[2] for r in results]

plt.figure(figsize=(10, 5))
plt.plot(n_vals, model_rmse, 'o-', label='Transformer')
plt.plot(n_vals, ols_rmse, 's--', label='OLS')
plt.xlabel('Context Length (n_ctx)')
plt.ylabel('RMSE')
plt.title('Model vs OLS Performance by Context Length')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Full evaluation suite
print("\nFull Evaluation Suite (n_ctx=64)")
print("=" * 50)

suite_results = eval_suite(
    model,
    d=CONFIG['d'],
    n_ctx=64,
    batch_size=1024,
    device=device,
)

## Load Checkpoint (Optional)

Run this cell to load a previously saved model.

In [ ]:
# Load a saved checkpoint
checkpoint_path = os.path.join(PROJECT_PATH, 'model_checkpoint.pt')

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    loaded_config = checkpoint['config']
    
    # Recreate model with saved config
    model = create_model(
        d=loaded_config['d'],
        n_ctx_max=loaded_config['n_ctx_max'],
        d_model=loaded_config['d_model'],
        n_heads=loaded_config['n_heads'],
        n_layers=loaded_config['n_layers'],
        device=device,
    )
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded checkpoint from {checkpoint_path}")
    print(f"Config: {loaded_config}")
else:
    print(f"No checkpoint found at {checkpoint_path}")